# Notebook 1: Inference-Only (Groq API)

## What this notebook does

This notebook answers Austrian tax law questions by sending each question to the
Groq API, which runs **Llama 3.3 70B** on Groq's dedicated inference hardware.

**Output:** `/content/inference_submission.csv` with columns `id` and `answer`.

---

## Before running

1. Upload `dataset_clean.csv` to `/content/`
2. Set `GROQ_API_KEY` in Colab Secrets (Settings icon at left -> Secrets -> + Add new secret)

**Estimated runtime: 22-28 minutes for 643 questions** (dominated by the rate limit, not the code)

## Step 1: Install dependencies

In [ ]:
!pip install -q groq

## Step 2: Load API key and dataset

In [ ]:
import os
import re
import time
import json
import pandas as pd

# Load Groq API key from Colab Secrets (preferred) or environment variable
try:
    from google.colab import userdata
    GROQ_KEY = userdata.get("GROQ_API_KEY") or ""
except Exception:
    GROQ_KEY = os.getenv("GROQ_API_KEY", "")

print(f"Groq key: {'SET' if GROQ_KEY else 'NOT SET -- set GROQ_API_KEY in Colab Secrets'}")

# Load dataset
df = pd.read_csv("/content/dataset_clean.csv")
print(f"\nLoaded {len(df)} questions.")
print(df[["id", "prompt"]].head(3).to_string())

## Step 3: Configuration

All tuneable settings are defined here so they are easy to find and adjust.

**Checkpoint file:** `inference_checkpoint.json` records every successfully answered question.
If the notebook crashes, re-run from Step 6 to pick up where it left off.
Delete the checkpoint file only if you want to start the full run from scratch.

**Note on error handling:** Questions that returned errors in a previous run are *not* stored
in the checkpoint. They will be retried automatically on resume.

In [ ]:
# File paths
CHECKPOINT_FILE = "/content/inference_checkpoint.json"
OUTPUT_CSV      = "/content/inference_submission.csv"
CHECKPOINT_EVERY = 50   # write checkpoint every N questions

# Model
GROQ_MODEL = "llama-3.3-70b-versatile"

# Prompt settings
# The system message is in German and instructs the model to be concise
# and to cite the relevant law where possible. This matches the task format.
SYSTEM_MSG = (
    "Du bist ein Experte für österreichisches Steuerrecht. "
    "Beantworte die folgende Frage kurz und präzise auf Deutsch. "
    "Nenne die relevante Gesetzesgrundlage, wenn möglich."
)

# 250 tokens is enough for a concise factual answer.
# Keeping this low reduces per-call latency.
MAX_TOKENS  = 150
TEMPERATURE = 0.1   # near-deterministic; we want factual answers

# Rate limit handling
# Groq free tier: ~30 req/min. Sleeping 2.1s between calls stays safely under the limit.
INTER_CALL_SLEEP = 3.5
MAX_RETRIES      = 4

print("Configuration loaded.")
print(f"  Model:             {GROQ_MODEL}")
print(f"  Max tokens:        {MAX_TOKENS}")
print(f"  Inter-call sleep:  {INTER_CALL_SLEEP}s")
print(f"  Checkpoint every:  {CHECKPOINT_EVERY} questions")
print(f"\nEstimated minimum runtime: {len(df) * INTER_CALL_SLEEP / 60:.1f} min (rate-limit bound)")

## Step 4: Define the API call function

One function handles one question with retry logic:

- On rate-limit errors (429): reads the `retry-after` value from the error if available,
  otherwise backs off with increasing waits (15s, 30s, 45s, 60s)
- On invalid API key: returns an error immediately without retrying
- On other errors: waits 3 seconds and retries
- On complete failure: returns `None` so the main loop knows to skip checkpointing

Returning `None` on failure (rather than an error string) is important:
it ensures failed questions are not written to the checkpoint and will be retried on resume.

In [ ]:
def call_groq(question, groq_client):
    """
    Send one question to Groq. Returns the answer string on success, or None on failure.
    Returning None (not an error string) means the question will be retried on resume.
    """
    for attempt in range(MAX_RETRIES):
        try:
            response = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_MSG},
                    {"role": "user",   "content": question}
                ],
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE
            )
            return response.choices[0].message.content.strip()

        except Exception as e:
            err = str(e).lower()

            if "invalid_api_key" in err or "authentication" in err:
                # Key problem: no point retrying
                print(f"  [Auth error] Check your API key: {e}")
                return None

            if "rate_limit" in err or "429" in err:
                # Try to read the retry-after value from the error message
                wait = 15 * (attempt + 1)   # fallback: 15s, 30s, 45s, 60s
                m = re.search(r"retry.after[^\d]*(\d+)", str(e), re.I)
                if m:
                    wait = int(m.group(1)) + 2
                print(f"  [Rate limit] Waiting {wait}s before retry {attempt+1}/{MAX_RETRIES}...")
                time.sleep(wait)
            else:
                # Other error: short wait and retry
                print(f"  [Error attempt {attempt+1}] {str(e)[:80]}")
                time.sleep(3)

    print(f"  [Failed] All {MAX_RETRIES} retries exhausted.")
    return None


print("API call function defined.")

## Step 5: Validate API key with one test call

We send one test question before starting the full run.
If the key is wrong or the quota is exhausted, we stop here rather than waste 20+ minutes.

In [ ]:
from groq import Groq

if not GROQ_KEY:
    raise RuntimeError(
        "GROQ_API_KEY not found. "
        "Go to Settings > Secrets, add GROQ_API_KEY, then re-run Steps 2 and 5."
    )

groq_client = Groq(api_key=GROQ_KEY)

print("Testing Groq API...")
test = call_groq("Was ist die Körperschaftsteuer in Österreich?", groq_client)

if test is None:
    raise RuntimeError(
        "Groq test call failed. Check your API key and quota, then re-run."
    )

print(f"Groq OK. Preview: {test[:120]}...")

## Step 6: Load checkpoint

The checkpoint file stores all questions that were answered successfully in previous runs.
Only successfully answered questions are stored -- failed ones are not.
This means re-running from this step automatically retries any previously failed questions.

In [ ]:
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        checkpoint = json.load(f)   # dict: {str(id): answer}
    print(f"Checkpoint loaded: {len(checkpoint)} answers already done.")
else:
    checkpoint = {}
    print("No checkpoint found. Starting fresh.")

ids       = df["id"].tolist()
questions = df["prompt"].tolist()

# Only process questions not yet in the checkpoint
remaining = [
    (ids[i], questions[i])
    for i in range(len(df))
    if str(ids[i]) not in checkpoint
]

print(f"Remaining to process: {len(remaining)} / {len(df)} questions")

## Step 7: Run inference

The loop is sequential and explicit:

- One question per API call
- Only successful answers are written to the checkpoint
- Progress and ETA are printed every 50 questions
- If 8 consecutive questions fail, the run stops and saves the checkpoint

**Expected runtime: 22-28 minutes for 643 questions.**
This is determined by the Groq free-tier rate limit, not by the code.
The code itself adds only a few seconds of overhead over the full run.

In [ ]:
def save_checkpoint():
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump(checkpoint, f, ensure_ascii=False)


start_time         = time.time()
consecutive_errors = 0

for idx, (qid, question) in enumerate(remaining):

    answer = call_groq(question, groq_client)

    if answer is not None:
        # Success: store the answer and reset the error counter
        checkpoint[str(qid)] = answer
        consecutive_errors = 0
    else:
        # Failure: do not store in checkpoint so this question is retried on resume
        consecutive_errors += 1

    # If 8 questions fail in a row, the API is likely broken -- stop and save
    if consecutive_errors >= 8:
        save_checkpoint()
        raise RuntimeError(
            f"8 consecutive failures at question index {idx+1}. "
            f"Check your API key or daily quota. "
            f"Checkpoint saved with {len(checkpoint)} answers."
        )

    # Save checkpoint and print progress every N questions
    if (idx + 1) % CHECKPOINT_EVERY == 0 or (idx + 1) == len(remaining):
        save_checkpoint()
        elapsed       = time.time() - start_time
        rate          = (idx + 1) / elapsed * 60    # questions per minute
        remaining_q   = len(remaining) - (idx + 1)
        eta           = (remaining_q / (idx + 1)) * elapsed / 60 if idx > 0 else 0
        print(
            f"Progress: {idx+1}/{len(remaining)} | "
            f"Answered so far: {len(checkpoint)} | "
            f"Rate: {rate:.1f} req/min | ETA: {eta:.1f} min"
        )

    # Sleep between calls to stay within the rate limit
    # Skip sleep after the last question and after failures
    # (failures already waited during retry backoff)
    if idx < len(remaining) - 1 and answer is not None:
        time.sleep(INTER_CALL_SLEEP)


elapsed_total = time.time() - start_time
print(f"\nInference complete: {elapsed_total:.0f}s ({elapsed_total/60:.1f} min)")
print(f"Checkpoint contains {len(checkpoint)} successful answers.")

## Step 8: Build and save the final CSV

Answers are assembled in the original question order.
Any question not in the checkpoint (i.e. failed after all retries) receives a placeholder answer.
The placeholder is included so the output always has the same number of rows as the input.

In [ ]:
final_answers = [
    checkpoint.get(str(qid), "Keine Antwort verfügbar.")
    for qid in df["id"].tolist()
]

submission = pd.DataFrame({
    "id":     df["id"].tolist(),
    "answer": final_answers
})

submission.to_csv(OUTPUT_CSV, index=False)

missing = sum(1 for a in final_answers if a == "Keine Antwort verfügbar.")
print(f"Saved: {OUTPUT_CSV}")
print(f"  Total rows:   {len(submission)}")
print(f"  Answered:     {len(submission) - missing}")
print(f"  Missing:      {missing}")
print()
print("Sample answers:")
for _, row in submission.head(3).iterrows():
    print(f"  ID {row['id']}: {str(row['answer'])[:100]}...")

## Step 9: Verify output

Reload the saved CSV and confirm row count and column names.
If missing answers are found, re-run from Step 6 to retry them.

In [ ]:
check = pd.read_csv(OUTPUT_CSV)

print(f"Row count: {len(check)} (expected {len(df)}) -- {'OK' if len(check) == len(df) else 'MISMATCH'}")
print(f"Columns:   {list(check.columns)}")

missing = (check["answer"] == "Keine Antwort verfügbar.").sum()
if missing > 0:
    print(f"\nWARNING: {missing} questions have no answer.")
    print("Re-run from Step 6 to retry them.")
else:
    print("\nAll questions answered. File looks good.")